# Color Attenuation Prior (CAP)
Zhu, Mai, Shao (IEEE TIP 2015). Physics prior like DCP, but estimates depth from brightness and saturation: haze raises brightness and lowers saturation, so their difference grows with distance. Training-free.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE     = "/content/drive/MyDrive"
SRC       = f"{DRIVE}/O-HAZE"          # original O-HAZE: hazy/ + GT/
DATA_ROOT = f"{DRIVE}/O-HAZE-1600"     # resized copy, shared by all methods and the evaluation
OUT       = f"{DRIVE}/outputs/cap" # dehazed images, same file names as hazy/

MAX_IMAGES = 5             # first N pairs (None = all 45)
RESIZE = 1600              # long side; O-HAZE is ~3000x4000

In [ ]:
#resize hazy + GT once (skipped if DATA_ROOT already exists), same as the other method notebooks
import os, glob
from PIL import Image

def resize_long(img, long_side):
    if long_side <= 0 or max(img.size) <= long_side:
        return img
    s = long_side / max(img.size)
    return img.resize((round(img.size[0] * s), round(img.size[1] * s)), Image.BICUBIC)

if not os.path.isdir(f"{DATA_ROOT}/hazy"):
    os.makedirs(f"{DATA_ROOT}/hazy"); os.makedirs(f"{DATA_ROOT}/GT")
    pairs = list(zip(sorted(glob.glob(f"{SRC}/hazy/*")), sorted(glob.glob(f"{SRC}/GT/*"))))[:MAX_IMAGES]
    for h, g in pairs:
        hazy = resize_long(Image.open(h).convert('RGB'), RESIZE)
        gt   = Image.open(g).convert('RGB').resize(hazy.size, Image.BICUBIC)
        stem = lambda p: os.path.splitext(os.path.basename(p))[0]
        hazy.save(f"{DATA_ROOT}/hazy/{stem(h)}.png"); gt.save(f"{DATA_ROOT}/GT/{stem(g)}.png")

hazy_paths = sorted(glob.glob(f"{DATA_ROOT}/hazy/*"))
os.makedirs(OUT, exist_ok=True)
print(len(hazy_paths), "images:", [os.path.basename(p) for p in hazy_paths])

## Parameters

In [ ]:
# linear depth model d = T0 + T1 * value + T2 * saturation, coefficients learned in the paper
T0, T1, T2 = 0.121779, 0.959710, -0.780245
BETA = 1.0                 # scattering coefficient: t = exp(-BETA * d)
PATCH = 15                 # min filter on depth (same role as the dark channel patch)
TOP_PERCENT = 0.001        # deepest 0.1% pixels are candidates for the atmospheric light A
T_MIN, T_MAX = 0.1, 0.9    # transmission bounds, as in the paper's reference implementation

# depth refinement with a guided filter
GF_RADIUS = 60
GF_EPS = 1e-3

NAME = "cap"

## Method

In [ ]:
def guided_filter(guide, src, r=GF_RADIUS, eps=GF_EPS):
    #edge-preserving smoothing of src, following the edges of guide (He et al., 2010)
    box = lambda x: cv2.boxFilter(x, -1, (2 * r + 1, 2 * r + 1))
    mean_I, mean_p = box(guide), box(src)
    a = (box(guide * src) - mean_I * mean_p) / (box(guide * guide) - mean_I * mean_I + eps)
    b = mean_p - a * mean_I
    return box(a) * guide + box(b)


def dehaze(bgr):
    I = bgr.astype(np.float32) / 255
    hsv = cv2.cvtColor(I, cv2.COLOR_BGR2HSV)        # float input: S, V in [0, 1]
    d = T0 + T1 * hsv[..., 2] + T2 * hsv[..., 1]    # depth from brightness and saturation

    d = cv2.erode(d, cv2.getStructuringElement(cv2.MORPH_RECT, (PATCH, PATCH)))   # local min
    d = guided_filter(cv2.cvtColor(I, cv2.COLOR_BGR2GRAY), d.astype(np.float32))

    #A: among the deepest pixels, the brightest one in the input
    n = max(int(d.size * TOP_PERCENT), 1)
    cand = np.argpartition(d.ravel(), -n)[-n:]
    flat = I.reshape(-1, 3)
    A = flat[cand[flat[cand].sum(axis=1).argmax()]]

    t = np.clip(np.exp(-BETA * d), T_MIN, T_MAX)
    J = (I - A) / t[..., None] + A                  # invert I = J*t + A*(1-t)
    return (np.clip(J, 0, 1) * 255 + 0.5).astype(np.uint8)

## Preview a single image

In [ ]:
hazy = cv2.imread(hazy_paths[0])
out = dehaze(hazy)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, im, title in zip(ax, [hazy, out], ['hazy', NAME]):
    a.imshow(im[..., ::-1]); a.set_title(title); a.axis('off')
plt.tight_layout(); plt.show()

## Run over the dataset

In [ ]:
import time
for p in hazy_paths:
    hazy = cv2.imread(p)
    t0 = time.time()
    out = dehaze(hazy)
    dt = time.time() - t0
    cv2.imwrite(f"{OUT}/{os.path.basename(p)}", out)
    print(f"{os.path.basename(p)} | {hazy.shape[1]}x{hazy.shape[0]} | {dt:.2f}s")

## Evaluate
Score the outputs with `dehazing/evaluation` (see `../DEHAZING_EVALUATION.ipynb`):
```python
per_image, summary = evaluate(DATA_ROOT, {"cap": OUT})
```